# FireWatch Indonesia - Exploratory Data Analysis (EDA)
## Analisis Eksplorasi Data Hotspot

**Tujuan:**
- Memahami distribusi dan pola data hotspot
- Menganalisis tren temporal (waktu)
- Menganalisis distribusi spasial (geografis)
- Mengidentifikasi insight awal dari data

## 1. Import Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi visualisasi
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10
%matplotlib inline

print("✓ Library berhasil dimuat")

In [ ]:
# Load dataset
df = pd.read_csv('../data/raw/fire_nrt_SV-C2_800747.csv')
df['acq_date'] = pd.to_datetime(df['acq_date'])

print(f"Dataset: {len(df):,} baris")
print(f"Periode: {df['acq_date'].min()} s/d {df['acq_date'].max()}")

In [ ]:
# Preview data
df.head()

## 2. Statistik Deskriptif

In [ ]:
# Statistik untuk variabel numerik
print("Statistik Deskriptif Variabel Numerik:")
print("="*80)
df[['brightness', 'frp', 'latitude', 'longitude']].describe()

## 3. Analisis Temporal (Tren Waktu)

### 3.1 Tren Harian

In [ ]:
# Tren harian
daily_hotspot = df.groupby('acq_date').size().reset_index(name='jumlah_hotspot')

print(f"Rata-rata hotspot/hari: {daily_hotspot['jumlah_hotspot'].mean():.0f}")
print(f"Hotspot tertinggi: {daily_hotspot['jumlah_hotspot'].max():,} pada {daily_hotspot.loc[daily_hotspot['jumlah_hotspot'].idxmax(), 'acq_date'].strftime('%Y-%m-%d')}")
print(f"Hotspot terendah: {daily_hotspot['jumlah_hotspot'].min():,} pada {daily_hotspot.loc[daily_hotspot['jumlah_hotspot'].idxmin(), 'acq_date'].strftime('%Y-%m-%d')}")

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(daily_hotspot['acq_date'], daily_hotspot['jumlah_hotspot'], 
         marker='o', linewidth=2, markersize=4, color='#e74c3c')
plt.xlabel('Tanggal', fontsize=12, fontweight='bold')
plt.ylabel('Jumlah Hotspot', fontsize=12, fontweight='bold')
plt.title('Tren Harian Deteksi Hotspot di Indonesia', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Tren Mingguan

In [ ]:
# Tren per minggu
df['week'] = df['acq_date'].dt.to_period('W')
weekly_hotspot = df.groupby('week').size().reset_index(name='jumlah_hotspot')
weekly_hotspot['week_str'] = weekly_hotspot['week'].astype(str)

plt.figure(figsize=(12, 6))
plt.bar(range(len(weekly_hotspot)), weekly_hotspot['jumlah_hotspot'], color='#3498db', alpha=0.8)
plt.xlabel('Minggu', fontsize=12, fontweight='bold')
plt.ylabel('Jumlah Hotspot', fontsize=12, fontweight='bold')
plt.title('Distribusi Hotspot per Minggu', fontsize=14, fontweight='bold')
plt.xticks(range(len(weekly_hotspot)), weekly_hotspot['week_str'], rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 3.3 Distribusi per Hari dalam Seminggu

In [ ]:
# Distribusi per hari dalam seminggu
df['day_of_week'] = df['acq_date'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_hotspot = df['day_of_week'].value_counts().reindex(day_order)

plt.figure(figsize=(10, 6))
dow_hotspot.plot(kind='bar', color='#2ecc71', alpha=0.8)
plt.xlabel('Hari', fontsize=12, fontweight='bold')
plt.ylabel('Jumlah Hotspot', fontsize=12, fontweight='bold')
plt.title('Distribusi Hotspot Berdasarkan Hari dalam Seminggu', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Analisis Confidence Level

In [ ]:
# Distribusi confidence
confidence_dist = df['confidence'].value_counts()
confidence_labels = {'n': 'Normal', 'l': 'Low', 'h': 'High'}
confidence_dist.index = confidence_dist.index.map(confidence_labels)

print("Distribusi Confidence Level:")
for conf, count in confidence_dist.items():
    pct = (count/len(df))*100
    print(f"  {conf:10s}: {count:7,} ({pct:5.2f}%)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
confidence_dist.plot(kind='bar', ax=ax1, color=['#3498db', '#e74c3c', '#2ecc71'], alpha=0.8)
ax1.set_xlabel('Confidence Level', fontsize=12, fontweight='bold')
ax1.set_ylabel('Jumlah Hotspot', fontsize=12, fontweight='bold')
ax1.set_title('Distribusi Confidence Level', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Pie chart
colors = ['#3498db', '#e74c3c', '#2ecc71']
ax2.pie(confidence_dist.values, labels=confidence_dist.index, autopct='%1.1f%%', 
        startangle=90, colors=colors, textprops={'fontsize': 11})
ax2.set_title('Proporsi Confidence Level', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Analisis Day vs Night

In [ ]:
# Distribusi day vs night
daynight_dist = df['daynight'].value_counts()
daynight_labels = {'D': 'Day (Siang)', 'N': 'Night (Malam)'}
daynight_dist.index = daynight_dist.index.map(daynight_labels)

print("Distribusi Day/Night:")
for dn, count in daynight_dist.items():
    pct = (count/len(df))*100
    print(f"  {dn:15s}: {count:7,} ({pct:5.2f}%)")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
daynight_dist.plot(kind='bar', ax=ax1, color=['#f39c12', '#34495e'], alpha=0.8)
ax1.set_xlabel('Waktu Deteksi', fontsize=12, fontweight='bold')
ax1.set_ylabel('Jumlah Hotspot', fontsize=12, fontweight='bold')
ax1.set_title('Distribusi Deteksi Siang vs Malam', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Pie chart
ax2.pie(daynight_dist.values, labels=daynight_dist.index, autopct='%1.1f%%',
        startangle=90, colors=['#f39c12', '#34495e'], textprops={'fontsize': 11})
ax2.set_title('Proporsi Deteksi Siang vs Malam', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Analisis Brightness & FRP

In [ ]:
print("Statistik Brightness & FRP:")
print(f"Brightness - Mean: {df['brightness'].mean():.2f} K, Median: {df['brightness'].median():.2f} K")
print(f"FRP - Mean: {df['frp'].mean():.2f} MW, Median: {df['frp'].median():.2f} MW")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Brightness distribution
axes[0,0].hist(df['brightness'], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
axes[0,0].set_xlabel('Brightness (K)', fontweight='bold')
axes[0,0].set_ylabel('Frequency', fontweight='bold')
axes[0,0].set_title('Distribusi Brightness', fontweight='bold')
axes[0,0].grid(alpha=0.3)

# FRP distribution
axes[0,1].hist(df['frp'], bins=50, color='#3498db', alpha=0.7, edgecolor='black')
axes[0,1].set_xlabel('FRP (MW)', fontweight='bold')
axes[0,1].set_ylabel('Frequency', fontweight='bold')
axes[0,1].set_title('Distribusi Fire Radiative Power (FRP)', fontweight='bold')
axes[0,1].grid(alpha=0.3)

# Brightness boxplot
axes[1,0].boxplot(df['brightness'], vert=True)
axes[1,0].set_ylabel('Brightness (K)', fontweight='bold')
axes[1,0].set_title('Boxplot Brightness', fontweight='bold')
axes[1,0].grid(alpha=0.3)

# FRP boxplot
axes[1,1].boxplot(df['frp'], vert=True)
axes[1,1].set_ylabel('FRP (MW)', fontweight='bold')
axes[1,1].set_title('Boxplot FRP', fontweight='bold')
axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Distribusi Geografis

In [ ]:
print(f"Koordinat:")
print(f"Latitude: {df['latitude'].min():.2f}° hingga {df['latitude'].max():.2f}°")
print(f"Longitude: {df['longitude'].min():.2f}° hingga {df['longitude'].max():.2f}°")

In [ ]:
# Scatter plot
plt.figure(figsize=(14, 8))
plt.scatter(df['longitude'], df['latitude'], alpha=0.1, s=1, c='red')
plt.xlabel('Longitude', fontsize=12, fontweight='bold')
plt.ylabel('Latitude', fontsize=12, fontweight='bold')
plt.title('Distribusi Geografis Hotspot di Indonesia', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap density
plt.figure(figsize=(14, 8))
plt.hexbin(df['longitude'], df['latitude'], gridsize=50, cmap='YlOrRd', mincnt=1)
plt.colorbar(label='Jumlah Hotspot')
plt.xlabel('Longitude', fontsize=12, fontweight='bold')
plt.ylabel('Latitude', fontsize=12, fontweight='bold')
plt.title('Heatmap Density Hotspot di Indonesia', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Top Lokasi Hotspot

In [ ]:
# Buat grid koordinat (0.5 derajat)
df['lat_grid'] = (df['latitude'] // 0.5) * 0.5
df['lon_grid'] = (df['longitude'] // 0.5) * 0.5
df['grid_cell'] = df['lat_grid'].astype(str) + ',' + df['lon_grid'].astype(str)

top_locations = df['grid_cell'].value_counts().head(10)

print("Top 10 Grid Cell (0.5° x 0.5°) dengan Hotspot Terbanyak:")
for i, (grid, count) in enumerate(top_locations.items(), 1):
    lat, lon = grid.split(',')
    print(f"  {i:2d}. Lat {float(lat):7.2f}°, Lon {float(lon):7.2f}° : {count:6,} hotspot")

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(range(len(top_locations)), top_locations.values, color='#e67e22', alpha=0.8)
plt.xlabel('Grid Cell', fontsize=12, fontweight='bold')
plt.ylabel('Jumlah Hotspot', fontsize=12, fontweight='bold')
plt.title('Top 10 Lokasi dengan Hotspot Terbanyak', fontsize=14, fontweight='bold')
plt.xticks(range(len(top_locations)), [f"#{i+1}" for i in range(len(top_locations))])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Summary EDA

In [ ]:
print("=" * 70)
print("SUMMARY EDA - FIREWATCH INDONESIA")
print("=" * 70)
print(f"\nTotal Hotspot: {len(df):,}")
print(f"Periode: {df['acq_date'].min().strftime('%Y-%m-%d')} s/d {df['acq_date'].max().strftime('%Y-%m-%d')}")
print(f"\nRata-rata hotspot per hari: {daily_hotspot['jumlah_hotspot'].mean():.0f}")
print(f"Puncak hotspot: {daily_hotspot['jumlah_hotspot'].max():,} pada {daily_hotspot.loc[daily_hotspot['jumlah_hotspot'].idxmax(), 'acq_date'].strftime('%Y-%m-%d')}")
print(f"\nLokasi dengan konsentrasi tertinggi:")
for i, (grid, count) in enumerate(list(top_locations.items())[:3], 1):
    lat, lon = grid.split(',')
    print(f"  {i}. Lat {float(lat):.2f}°, Lon {float(lon):.2f}° : {count:,} hotspot")
print("\n" + "=" * 70)

## Key Findings

Berdasarkan analisis eksplorasi data:

1. **Tren Temporal:**
   - Aktivitas hotspot menunjukkan variasi harian yang signifikan
   - Terdapat periode dengan peningkatan deteksi hotspot yang tajam

2. **Confidence Level:**
   - Mayoritas deteksi memiliki confidence level normal (>90%)
   - Menunjukkan kualitas data yang baik

3. **Waktu Deteksi:**
   - Lebih banyak hotspot terdeteksi di siang hari
   - Kemungkinan terkait aktivitas pembakaran lahan atau kondisi cuaca

4. **Distribusi Geografis:**
   - Konsentrasi tinggi di wilayah Kalimantan Tengah dan Selatan
   - Pola geografis menunjukkan area-area kritis yang perlu perhatian

**Langkah selanjutnya:**
- Analisis spasial lebih detail per provinsi
- Integrasi dengan data cuaca
- Pengembangan dashboard interaktif